[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1pnUO5jv55vXK9dtyNil_MLI7Bd_q2W96)

In [ ]:
# Setup

!pip install -q langchain langchain-groq

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [ ]:
# The assistant: template + memory + chain

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Satwik. Do you make eggless cakes?"))
print(ask("What's my name?"))

Yes, we offer egg‑less versions of all our cakes – plum, chocolate, and cupcakes. Just give us a 24‑hour notice and we’ll have your order ready. Let me know what you’d like!
Your name is Satwik.


In [ ]:
# Tests

import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()      # create tests.csv first with the %%writefile cell from topic 8

PASS | How much is a plum cake? → A plum cake costs ₹500. Let me know if you’d like to place an order!
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes – plum, chocolate, and
FAIL | Can I get a cake today evening? → I’m sorry, we need 24‑hour notice for cake orders. I’ll check with the
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | What is the price of 6 cupcakes? → 6 cupcakes cost Rs 480.

4/5 passed (80%)


In [ ]:
# Sneak peek: the same assistant as a LangChain agent

from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes, all our cakes—plum, chocolate, and cupcakes—have eggless versions. Feel free to let us know your preference!


In [ ]:
# TASK 1: Rewrite SYSTEM for the business you chose and change the {business} details.

SYSTEM = f"""ROLE:   You are the helpful front-desk assistant for Fitness Studio.
FACTS:  Packages & prices in rupees:
          Morning yoga pass Rs 1500,
          Strength batch Rs 2200,
          Weekend sound bath Rs 600,
        Morning batches start at 06:30 AM. Mats are provided free.
RULES:  Never guarantee spot availability or exact class end times.
        If it isn't in FACTS, say "I'll check with the studio manager."
FORMAT: Under 50 words, warm and clear, matching the member's language."""

fitness_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{question}"),
])
fitness_chain = fitness_prompt | llm | StrOutputParser()

In [ ]:
# TASK 2: Turn your 10 customer questions into tests.csv with a must_include for each.
# Run them and write down the pass rate.

import csv

def run_tests(path: str = "/content/tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = fitness_chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()      # create tests.csv first with the %%writefile cell from topic 8

PASS | How much is the Morning Yoga Pass? → The Morning Yoga Pass is Rs 1500. Mats are provided free. Let me know 
PASS | What is the price of the Strength Batch? → The Strength Batch costs Rs 2200. Let me know if you’d like to book a 
PASS | What is the price of the Weekend Sound Bath? → The Weekend Sound Bath is Rs 600.
PASS | What time do morning classes start? → Morning classes start at 06:30 AM. We’ll have mats ready for you. Let 
PASS | Do I need to bring my own yoga mat? → No, you don’t need to bring your own. We provide free mats for all mor
PASS | Can you tell me the evening batch timings? → I’m not sure about the evening batch timings. I’ll check with the stud
PASS | Do you offer personal training sessions? → I’ll check with the studio manager.
PASS | Is there a spot available in tomorrow's yoga class? → I’ll check with the studio manager.
PASS | What time exactly will my morning yoga class finish? → I’m not sure of the exact finish time. I’ll check with the studio mana
PASS 

In [ ]:
# TASK 3: change one thing in the prompt, run the tests again, and note whether the pass rate went up or down.

#Change: Matss are free -> Mats are not provided.
SYSTEM = f"""ROLE:   You are the helpful front-desk assistant for Fitness Studio.
FACTS:  Packages & prices in rupees:
          Morning yoga pass Rs 1500,
          Strength batch Rs 2200,
          Weekend sound bath Rs 600,
        Morning batches start at 06:30 AM. Mats are not provided.
RULES:  Never guarantee spot availability or exact class end times.
        If it isn't in FACTS, say "I'll check with the studio manager."
FORMAT: Under 50 words, warm and clear, matching the member's language."""

fitness_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{question}"),
])
fitness_chain = fitness_prompt | llm | StrOutputParser()

In [ ]:
import csv

def run_tests(path: str = "/content/tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = fitness_chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()      # create tests.csv first with the %%writefile cell from topic 8

FAIL | How much is the Morning Yoga Pass? → The Morning Yoga Pass is Rs 1,500.
FAIL | What is the price of the Strength Batch? → The Strength batch costs Rs 2,200.
PASS | What is the price of the Weekend Sound Bath? → The Weekend Sound Bath is ₹600. Let me know if you’d like to book a sp
PASS | What time do morning classes start? → Morning batches start at 06:30 AM. Let me know if you’d like to book a
FAIL | Do I need to bring my own yoga mat? → Yes, you’ll need to bring your own yoga mat. We don’t provide mats, so
PASS | Can you tell me the evening batch timings? → I’m not sure about the evening batch timings. I’ll check with the stud
PASS | Do you offer personal training sessions? → I’ll check with the studio manager.
PASS | Is there a spot available in tomorrow's yoga class? → I’ll check with the studio manager.
PASS | What time exactly will my morning yoga class finish? → I’m not sure of the exact finish time; morning yoga usually lasts abou
PASS | Can I join a weekend sound bath, 

In [ ]:
# TASK 4: change temperature to 1.0 and run the tests twice. Do the results change between runs?

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=1.0)

In [ ]:
run_tests()

PASS | How much is the Morning Yoga Pass? → The Morning Yoga Pass is Rs 1500.
PASS | What is the price of the Strength Batch? → The Strength batch is priced at Rs 2200. Let me know if you’d like to 
PASS | What is the price of the Weekend Sound Bath? → The Weekend Sound Bath is priced at Rs 600. Enjoy the relaxing experie
PASS | What time do morning classes start? → Morning classes start at 06:30 AM. We’ll have mats ready for you. Let 
PASS | Do I need to bring my own yoga mat? → No need to bring your own mat—mats are provided free for all yoga sess
PASS | Can you tell me the evening batch timings? → I’m sorry, I don’t have that info. I’ll check with the studio manager 
PASS | Do you offer personal training sessions? → I’m not sure about personal training sessions. I’ll check with the stu
PASS | Is there a spot available in tomorrow's yoga class? → I'll check with the studio manager.
PASS | What time exactly will my morning yoga class finish? → I’m not sure about the exact finish time.

In [ ]:
run_tests()

PASS | How much is the Morning Yoga Pass? → The Morning Yoga Pass is Rs 1500. Enjoy your practice!
PASS | What is the price of the Strength Batch? → The Strength Batch costs Rs 2200.
PASS | What is the price of the Weekend Sound Bath? → The Weekend Sound Bath is priced at ₹600. Enjoy the relaxing experienc
PASS | What time do morning classes start? → Morning classes start at 06:30 AM.
PASS | Do I need to bring my own yoga mat? → No need to bring your own mat—mats are provided free for all classes. 
PASS | Can you tell me the evening batch timings? → I’m not sure about the evening batch times. I’ll check with the studio
PASS | Do you offer personal training sessions? → I’m not sure about personal training sessions. I’ll check with the stu
PASS | Is there a spot available in tomorrow's yoga class? → I’ll check with the studio manager and let you know as soon as possibl
PASS | What time exactly will my morning yoga class finish? → I’ll check with the studio manager and get back to you.
PA

In [ ]:
# TASK 5: list three jobs in your business and label each: plain code, one LLM call, or agent.

# Calculate package totals - Simple Math Function
# Answer FAQ and pricing Questions - One LLM call
# Handle bookings - Agent

In [ ]:
# Before Day 4
# Write two Python functions your business would need,
# for example a price lookup and an "is it available?" check, with type hints and docstrings (Day 2). Tomorrow they become the agent's tools.

PRICES: dict[str, int] = {
    "morning yoga pass": 1500,
    "strength batch": 2200,
    "weekend sound bath": 600,
}

OPEN_SPOTS: dict[str, int] = {
    "morning yoga": 4,
    "strength": 2,
    "weekend sound bath": 0,
}


def get_price(package_name: str) -> int | None:
    """Return the price in rupees for a package, or None if not found."""
    return PRICES.get(package_name.strip().lower())


def is_available(batch_name: str) -> bool:
    """Return True if at least one spot is open in the batch, else False."""
    return OPEN_SPOTS.get(batch_name.strip().lower(), 0) > 0

print(get_price("morning yoga pass"))
print(is_available("weekend sound bath"))

1500
False
